# WRI Demo

The [**Wildfire Resilience Index (WRI)**](https://www.wildfireindex.org/) is an interactive tool designed to support communities and landscapes living with wildfire in 12 Western US states, British Columbia, and the Yukon Territory (see image on the right).

Wildfire is natural and inevitable across the western United States and Canada. Living with it requires a shared understanding of how systems-both ecological and human-resist harm and recover after fire. Together, these abilities define resilience.

## Import libraries

In [ ]:
import requests
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_geom, calculate_default_transform, reproject, Resampling
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import json, re
import contextily as ctx
import geopandas as gpd
from shapely.geometry import shape
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings("ignore")

## Domain Scores

Working with experts, the team identified eight key areas—called domains—that capture how communities and landscapes respond to wildfire: Infrastructure, Communities, Livelihoods, Sense of Place, Species, Habitats, Water, and Air.

Each domain reflects both:
- Current conditions (status)
- The ability to resist harm and recover after wildfire (resilience)
  
To learn more about the methodology to calculate these scores, visit [this site](https://www.wildfireindex.org/methodology).

We will download 4 of the calculated domains scores: Air Quality, Communities, Livelihoods and Water. We have chosen these domains given the size of the TIF files. To access the rest of the domain scores, visit [this page](https://knb.ecoinformatics.org/view/urn%3Auuid%3Ac62b0d69-995b-41e3-af44-edb61915d569).

## Download Data

In [ ]:
import requests

files = [
    {
        "url": "https://knb.ecoinformatics.org/knb/d1/mn/v2/object/urn%3Auuid%3A3ec95526-e92a-409f-98fc-3ce7fcda63e3",
        "path": "air_quality_domain_score.tif"
    },
    {
        "url": "https://knb.ecoinformatics.org/knb/d1/mn/v2/object/urn%3Auuid%3Aa4558641-bb91-4c65-bdee-afc6fbc741ec",
        "path": "communities_domain_score.tif"
    },
    {
        "url": "https://knb.ecoinformatics.org/knb/d1/mn/v2/object/urn%3Auuid%3Ad9e8e302-bdc7-442e-b215-22c4a24b9500",
        "path": "livelihoods_domain_score.tif"
    },
    {
        "url": "https://knb.ecoinformatics.org/knb/d1/mn/v2/object/urn%3Auuid%3A6b55a1c4-eb10-496f-a66e-c01ed98a05bb",
        "path": "water_domain_score.tif"
    },
]

def download(url, path):
    print(f"Downloading {path}...")
    response = requests.get(url, stream=True)
    response.raise_for_status()

    total = int(response.headers.get('content-length', 0))
    downloaded = 0

    with open(path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                print(f"\r  {downloaded / total:.1%} ({downloaded/1e6:.1f} / {total/1e6:.1f} MB)", end="")

    print(f"\n  Done → {path}\n")

for f in files:
    download(f["url"], f["path"])

## Domain Scores over Fire Perimeters

In this notebook we will overlay selected Western Wildfire Resilience Index (WRI) domain scores (Air Quality, Communities, Livelihoods, and Water) onto historical fire perimeters across Southern California, allowing us to examine how resilient these dimensions were in areas that have already burned.

Use the dropdowns below to explore different resilience dimensions across fires. If the dropdowns do not appear after running the next cell, refresh the page in the browser so the widgets show up. To learn more about the fires included in this analysis, visit [this site](https://aicw.wildfirecommons.org/fires?fire=palisades-eaton-2025).

**NOTE:** Some areas within the fire perimeter may appear blank due to missing data in the underlying raster.

In [ ]:
# ── Layer registry ────────────────────────────────────────────────────────────
LAYERS = {
    "Air Quality":  ("air_quality_domain_score.tif",  "YlOrRd"),
    "Communities":  ("communities_domain_score.tif",  "RdYlGn"),
    "Livelihoods":  ("livelihoods_domain_score.tif",  "YlGn"),
    "Water":        ("water_domain_score.tif",         "Blues"),
}

# ── Fire registry ─────────────────────────────────────────────────────────────
FIRES = {
    "Airport 2024":   "https://firemap.sdsc.edu/geoserver/WIFIRE/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=WIFIRE:view_historical_fires&cql_filter=fire_name=%27AIRPORT%27%20AND%20year=2024&outputFormat=text/javascript&format_options=callback:_sd1778692284177ox053eztuzg",
    "Palisades 2025": "https://firemap.sdsc.edu/geoserver/WIFIRE/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=WIFIRE:view_historical_fires&cql_filter=fire_name=%27Palisades%27%20AND%20year=2025&outputFormat=text/javascript&format_options=callback:_sd1778698331829msbel2f1d8l",
    "Eaton 2025":     "https://firemap.sdsc.edu/geoserver/WIFIRE/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=WIFIRE:view_historical_fires&cql_filter=fire_name=%27EATON%27%20AND%20year=2025&outputFormat=text/javascript&format_options=callback:_sd17786983318293pidmu37tkj",
    "Caldor 2021":    "https://firemap.sdsc.edu/geoserver/WIFIRE/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=WIFIRE:view_historical_fires&cql_filter=fire_name=%27CALDOR%27%20AND%20year=2021&outputFormat=text/javascript&format_options=callback:_sd1778704837871nwveil4ibu",
    "Border 2 2025":  "https://firemap.sdsc.edu/geoserver/WIFIRE/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=WIFIRE:view_historical_fires&cql_filter=fire_name=%27BORDER%202%27%20AND%20year=2025&outputFormat=text/javascript&format_options=callback:_sd17787048572557l1olu4saek",
}

# ── Fire cache ────────────────────────────────────────────────────────────────
fire_cache = {}

def fetch_fire(name):
    if name in fire_cache:
        return fire_cache[name]
    resp = requests.get(FIRES[name])
    json_str = re.sub(r'^[^(]+\(', '', resp.text).rstrip(');')
    geojson = json.loads(json_str)
    props = geojson["features"][0]["properties"]
    gdf = gpd.GeoDataFrame(
        [props],
        geometry=[shape(geojson["features"][0]["geometry"])],
        crs="EPSG:4326"
    ).to_crs("EPSG:3857")
    fire_cache[name] = (gdf, geojson["features"][0]["geometry"], props)
    return fire_cache[name]

# ── clip → reproject ──────────────────────────────────────────────────────────
def clip_and_reproject(path, geometry_4326):
    with rasterio.open(path) as src:
        geom_proj = transform_geom("EPSG:4326", src.crs, geometry_4326)
        clipped, clipped_transform = mask(src, [geom_proj], crop=True, nodata=np.nan, all_touched=True)
        data = clipped[0].astype(float)
        if src.nodata is not None:
            data[data == src.nodata] = np.nan

        h, w = data.shape
        dst_crs = "EPSG:3857"
        left   = clipped_transform.c
        top    = clipped_transform.f
        right  = left + clipped_transform.a * w
        bottom = top  + clipped_transform.e * h

        t3857, width, height = calculate_default_transform(
            src.crs, dst_crs, w, h,
            left=left, bottom=bottom, right=right, top=top
        )
        reprojected = np.full((height, width), np.nan, dtype=np.float32)
        reproject(
            source=data,
            destination=reprojected,
            src_transform=clipped_transform,
            src_crs=src.crs,
            dst_transform=t3857,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear,
            src_nodata=np.nan,
            dst_nodata=np.nan,
        )
        out_bounds = rasterio.transform.array_bounds(height, width, t3857)
        return reprojected, out_bounds

# ── Widgets ───────────────────────────────────────────────────────────────────
layer_dd = widgets.Dropdown(
    options=list(LAYERS.keys()),
    description="Layer:",
    style={"description_width": "auto"},
    layout=widgets.Layout(width="250px")
)
fire_dd = widgets.Dropdown(
    options=list(FIRES.keys()),
    description="Fire:",
    style={"description_width": "auto"},
    layout=widgets.Layout(width="250px")
)
out = widgets.Output()

def plot(_=None):
    layer_name = layer_dd.value
    fire_name  = fire_dd.value
    path, cmap = LAYERS[layer_name]

    with out:
        clear_output(wait=True)
        print(f"Loading {layer_name} clipped to {fire_name}...")

        gdf, geometry, props = fetch_fire(fire_name)
        data, (left, bottom, right, top) = clip_and_reproject(path, geometry)
        bounds = gdf.total_bounds

        fire_label = props.get('fire_name', fire_name)
        year       = props.get('year', '')

        fig, ax = plt.subplots(figsize=(10, 8))

        ctx.add_basemap(ax, crs="EPSG:3857", source=ctx.providers.CartoDB.Positron)

        im = ax.imshow(
            data,
            extent=[left, right, bottom, top],
            cmap=cmap,
            alpha=0.75,
            interpolation="nearest",
            origin="upper",
            norm=mcolors.Normalize(vmin=np.nanmin(data), vmax=np.nanmax(data))
        )

        gdf.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2)

        ax.set_xlim(bounds[0], bounds[2])
        ax.set_ylim(bounds[1], bounds[3])
        ax.set_title(f"{layer_name} — {fire_label} ({year})", fontsize=13, fontweight="bold")
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Score")
        plt.tight_layout()
        plt.show()

layer_dd.observe(plot, names="value")
fire_dd.observe(plot,  names="value")

display(widgets.HBox([layer_dd, fire_dd]), out)
plot()